# Pytorch Transforms

## 학습 내용
- nn.Module 상송
- nn.Flatten
- nn.Linear
- nn.ReLU
- nn.Sequential
- forward() 함수
- Softmax
- Model Parameters

In [2]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        # 부모 클래스(nn.Module) 초기화
        super().__init__()
        # 이미지를 1차원 벡터로 펼침
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            # 784개 입력 -> feature 512개 출력
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            # 첫 번째 레이어가 만든 특징 조합하여 정교한 특징 추출
            # ex) 끈을 보여 운동화일 가능성 높다.
            nn.Linear(512, 512),
            nn.ReLU(),
            # 512개 특징을 10개의 점수로 출력
            nn.Linear(512, 10)
        )
    
    def forward(self, x):
        # 이미지를 1차원 벡터로 펼침
        x = self.flatten(x)
        # 신경망을 통과시켜 클래스별 점수 계산
        logits = self.linear_relu_stack(x)
        # 예측 점수 반환
        return logits

In [5]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [6]:
# 랜덤 이미지 생성
# 1개 이미지 28 * 28 픽셀
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
# Softmax 각 출력을 0~1범위 합 1로 만들어서 확률로 해석이 가능
# dim=0 행 방향, dim=1 열 방향
pred_probab = nn.Softmax(dim=1)(logits)
# 각 행에서 가장 큰 값의 위치 반환
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([9], device='cuda:0')


In [11]:
# Mudule Layers
input_image = torch.rand(3, 28, 28)
print(input_image.size())

torch.Size([3, 28, 28])


In [12]:
# nn.Flatten
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [14]:
# nn.Linear
layer1 = nn.Linear(in_features=28 * 28, out_features=20)
# 은닉층
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [15]:
# nn.ReLU
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.2722, -0.0536, -0.1905,  0.0057, -0.2598, -0.8311,  0.8773,  0.1177,
         -0.2827,  0.1380,  0.1881,  0.0924,  0.6600, -0.1962,  0.5072,  0.4400,
         -0.8846, -0.1413,  0.1937, -0.1821],
        [ 0.0864, -0.0449, -0.1027,  0.1137, -0.6321, -0.5456,  0.4237, -0.1969,
         -0.4735, -0.0038,  0.3229, -0.0357,  0.1968,  0.1591,  0.2673,  0.6164,
         -0.9345, -0.1476,  0.1196,  0.0680],
        [-0.3458,  0.0162, -0.2232,  0.2370, -0.2255, -0.7698,  0.6250,  0.1000,
         -0.3466,  0.0185,  0.4119, -0.1686,  0.5991, -0.1708,  0.4604,  0.8120,
         -0.8914, -0.2298,  0.3134,  0.0685]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.2722, 0.0000, 0.0000, 0.0057, 0.0000, 0.0000, 0.8773, 0.1177, 0.0000,
         0.1380, 0.1881, 0.0924, 0.6600, 0.0000, 0.5072, 0.4400, 0.0000, 0.0000,
         0.1937, 0.0000],
        [0.0864, 0.0000, 0.0000, 0.1137, 0.0000, 0.0000, 0.4237, 0.0000, 0.0000,
         0.0000, 0.3229, 0.0000, 0.1968, 0.1591, 0.26

In [17]:
# nn.Sequential
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)

input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)

In [18]:
# nn.Sorfmax
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

In [ ]:
# Model Parameters
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Value: {param[:2]}\n")

# weight와 bias가 딥러닝이 학습하는 대상

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Value: tensor([[ 0.0073,  0.0034, -0.0221,  ..., -0.0156,  0.0070,  0.0137],
        [ 0.0195, -0.0076, -0.0278,  ..., -0.0356,  0.0196, -0.0165]],
       device='cuda:0', grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Value: tensor([0.0080, 0.0118], device='cuda:0', grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Value: tensor([[ 0.0061, -0.0005,  0.0321,  ...,  0.0268, -0.0181, -0.0210],
        [-0.0134, -0.0079, -0.0310,  ...,  0.0184, -0.0021, -0.0190]],
       device='cuda:0', grad_fn=<SliceBackwar

## 정리
- nn.Module을 상속 하여 모델을 구현한다.
- nn.Flatten은 이미지를 1차원 벡터로 변환한다.
- nn.Linear는 입력 특징을 새로운 특징 공간으로 변환한다.
- nn.ReLU는 음수를 0으로 만들어 비선형성을 추가한다.
- nn.Sequential은 여러 레이어를 순서대로 연결한다.
- Softmax는 클래스 점수를 확률로 변환한다.
- 모델은 weight와 bias를 학습하여 성능을 향상시킨다.